# Import Libraries

In [ ]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [ ]:
print(device)

In [ ]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [ ]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 8
learning_rate = 0.001
epochs = 10

set_random_seed(42) # seed for reproducibility

In [ ]:
# Load BDD100KPlus dataset
trainloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

In [ ]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in test set: {len(testloader)}")

# Initialize WeatherNet Model

In [ ]:
# wn_model: WeatherNet = WeatherNet()
wn_model: WeatherNet = WeatherNetPlusPlus()
# wn_model: WeatherNet = MtlWeatherNet()

# Print the model architecture
print(wn_model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(wn_model.parameters())

# Define loss function
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
saved_state = None # path to saved model state
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    wn_model.load_state_dict(torch.load(saved_state, weights_only=True))

# Model Training

In [ ]:
# Train the WeatherNet model and record losses
# Use the train function to train the wn_model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# logs to runs/WeatherNet/ or runs/WeatherNetPlusPlus/ or runs/MtlWeatherNet/
writer = SummaryWriter(log_dir=f"runs/{wn_model.name}")

# epochs=1
wn_train_loss_log, wn_test_loss_log = trainer.train(wn_model, optimizer, criterion, trainloader, testloader, epochs, device, writer=writer)

writer.flush()

# Visualize Results

First, ensure you're in the correct environment:

`conda env create -f environment.yml`

This will create a conda environment called *tensorboard*, which you can activate via `conda activate tensorboard`

Then, to visualize the results: 

`tensorboard --logdir=runs`

This will automatically and recursively scan through all run logs in the runs/ directory.